In [14]:
## set project root
import sys, os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

assets_dir = os.path.join(project_root, "assets")

In [15]:
assets_dir

'/Users/autofrankie/GitHub/Projects/Used Car Price Prediction: Predictive Modeling with Real-World Data/assets'

# Import Packages

We'll make use of the following packages:
- `numpy` is a package for scientific computing in python.
- `pandas` A powerful Python library for data manipulation and analysis.
- `seaborn` A data visualization library based on matplotlib.
- `scikit-learn` A comprehensive library for machine learning in Python.
- `kaggle` Using Kaggle API to download data.

In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import kagglehub
import math
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import logging
import importlib
import src.data_preprocessing
import src.eda
importlib.reload(src.data_preprocessing)
importlib.reload(src.eda)
from src.data_preprocessing import DataPreprocessor
from src.eda import NumericalEDA, CategoricalEDA
from typing import List, Optional, Union

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression



# Download Data

In [17]:
# Download latest version
path = kagglehub.dataset_download("taeefnajib/used-car-price-prediction-dataset")

print(os.listdir(path))

['used_cars.csv']


## Check data

In [19]:
## Check data
df = pd.read_csv(os.path.join(path, "used_cars.csv"))

# Data Preprocessing via pipeline

In [20]:
# Data Preprocessing via pipeline
dp = DataPreprocessor(df,assets_dir=assets_dir)
df_new = dp.preprocess()

COMPLETE: rename_column. -
 {'milage': 'mileage'}
No missing values found.
Data preprocessing completed successfully.


# EDA

In [21]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   brand                4009 non-null   object 
 1   model                4009 non-null   object 
 2   model_year           4009 non-null   int64  
 3   mileage              4009 non-null   float64
 4   fuel_type            4009 non-null   object 
 5   engine               4009 non-null   object 
 6   transmission         4009 non-null   object 
 7   ext_col              4009 non-null   object 
 8   int_col              4009 non-null   object 
 9   accident             4009 non-null   object 
 10  clean_title          4009 non-null   int64  
 11  price                4009 non-null   float64
 12  transmission_type    4009 non-null   object 
 13  transmission_speeds  4009 non-null   float64
 14  hp                   4009 non-null   float64
 15  liters               4009 non-null   f

Column names and new features are showing as expected.

In [22]:
df_ml = df_new[['brand', 'model', 'model_year', 'mileage', 'fuel_type',
                'clean_title','transmission_type', 'transmission_speeds', 
                'hp', 'liters','cylinders', 'ext_color_std', 'int_col_std', 
                'accident_reported', 'price']]

In [23]:
df_ml.head()

,brand,model,model_year,mileage,fuel_type,clean_title,transmission_type,transmission_speeds,hp,liters,cylinders,ext_color_std,int_col_std,accident_reported,price
0,Ford,Utility Police Interceptor Base,2013,51000.0,E85 Flex Fuel,1,Automatic,6.0,300.0,3.7,6.0,black,black,1,10300.0
1,Hyundai,Palisade SEL,2021,34742.0,Gasoline,1,Automatic,8.0,291.0,3.8,6.0,blue,gray,1,38005.0
2,Lexus,RX 350,2022,22372.0,Gasoline,0,Automatic,8.0,295.0,3.5,6.0,blue,black,0,54598.0
3,INFINITI,Q50 Hybrid Sport,2015,88900.0,Hybrid,1,Automatic,7.0,354.0,3.5,6.0,black,black,0,15500.0
4,Audi,Q3 45 S line Premium Plus,2021,9835.0,Gasoline,0,Automatic,8.0,228.0,2.0,4.0,white,black,0,34999.0


# Functions

In [ ]:
class MachineLearningPipeline:
    def __init__(self, dataframe, target_col):
        self.dataframe = dataframe
        self.target_col = target_col
        self.X_train = None
        self.X_test = None 
        self.y_train = None 
        self.y_test = None

    def split_data(self, test_size=0.2, random_state=42, stratify=False):
        
        ## Split features and target
        X = self.dataframe.drop(columns=[self.target_col])
        y = self.dataframe[self.target_col]

        stratify_arg = y if stratify and y.unique() < 20 else None # Only for classification

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=stratify_arg)
        return self.X_train, self.X_test, self.y_train, self.y_test
    
    def LinearRegressionPipeline(self):
        pass

# Split Data

# Training Data EDA

### Numerical Histogram

In [ ]:
cat_cols = ['brand', 'model', 'fuel_type', 'clean_title','transmission_type','ext_color_std', 'int_col_std', 'accident_reported']
target = 'price'

numericaleda = NumericalEDA(df_ml, cat_cols=cat_cols, target_col=target)
numericaleda.plot_box_and_hist_per_feature()

## Categorical Bar Plot

In [ ]:
cat_eda = CategoricalEDA(dataframe=df_ml, cat_cols=cat_cols)
cat_eda.plot_bar_and_normalized_bars()

# EDA for transformation